# SA - Variance based, without rain parameters

## Imports and files

In [1]:
import os
import pickle
import scipy.io as sio
import itertools

from datetime import datetime
from datetime import timedelta

import numpy as np
import pandas as pd
import geopandas as gpd

import contextily as ctx
from shapely.geometry import Polygon
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from datetime import datetime
from collections import namedtuple

from swmm_api.input_file import read_inp_file, SwmmInput, section_labels as sections
from swmm_api import read_inp_file, read_out_file, swmm5_run

## Functions

In [2]:
def load_outflow_obs_to_df(OBS_outflow_DATA_PATH):
    """ Takes OBSERVATION FILE with outflow data and makes pd Dataframe
    :param output_path: str, path name that have the outflow data
    :return: df_obs_outflow
    """
    df_obs_outflow = pd.read_csv (OBS_outflow_DATA_PATH + ".csv")
    df_obs_outflow.dropna(inplace = True) ; df_obs_outflow.rename(columns={'discharge_cms': 'OBS outflow [CMS]'} , inplace=True, errors='raise')
    df_obs_outflow["date_and_time"] =  pd.to_datetime(df_obs_outflow["date_and_time"], format='%d/%m/%Y %H:%M:%S')
    df_obs_outflow = df_obs_outflow.set_index('date_and_time')
    df_obs_outflow.drop(['Unnamed: 0'], axis=1, inplace=True)


    return df_obs_outflow
    
# df_outflow.to_pickle('df_outflow.pkl')

In [3]:
def data_5min_interpolate(df_to_interpolate):
    """ Takes df column and makes 5 min interpolation
    :param df_to_interpolate: pd, df with index datetime64 type
    :return: df_interpol
    """
    df_interpol = df_to_interpolate.resample('5min').mean()
#     df_interpol= df_interpol.interpolate(method='polynomial', order=3)
    df_interpol= df_interpol.interpolate(method='linear')
    return df_interpol

In [4]:
def load_sim_to_df(sim_output_path):
    """ Takes SWMM FILE with outflow data and makes pd Dataframe
    sim_output_path: str, file name that have the swmm outflow data
    return: sim_df
    """
    sim_df = read_out_file(sim_output_path).to_frame()['system'][''][['outflow', 'rainfall']]
    sim_df.rename(columns={'outflow':'SWMM outflow [CMS]', 'rainfall':'rainfall [mm/h]'}, inplace=True, errors='raise')
    sim_df.index = pd.to_datetime(sim_df.index)
    return sim_df

In [5]:
def observation_and_swmm_hydrograph(storm_df, date):
    """
    This function takes observation and SWMM model outflow and precipitation data and puts it in a hydrograph.
    Params:
     - storm_df: df, The DataFrame that includes the outflow and precipitation data
     - date: date of storm occurrence. Format: 'yyyy_mm_dd'
    Returns: a hydrograph plot for one subcatchment
    """
    
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.suptitle(storm_df.index.strftime('%d/%m/%Y')[0], fontsize=20)

    y_obs_outflow = storm_df['OBS outflow [CMS]']
    y_rainfall = storm_df['rainfall [mm/h]']
    y_sim_outflow = storm_df['SWMM outflow [CMS]']

    ax1 = sns.lineplot(ax=ax, data=y_obs_outflow, color='g', label='Observed outflow')
    ax2 = ax1.twinx()
#     ax2.set_ylabel('Rainfall ($mm/hr$)')

    sns.lineplot(ax=ax2, data=y_rainfall, color='b', label='Rainfall', alpha=0.4)
    ax2.fill_between(storm_df.index, 0, y_rainfall, alpha=0.4, color='b')
#     ax2.set_ylim(ymin=y_rainfall.max() + 50, ymax=0)
    ax2.set_ylim(ymin=70, ymax=0)
    ax1.set_ylim(ymin=0, ymax=90)
    

    sns.lineplot(ax=ax, data=y_sim_outflow, color='r', label='Simulated outflow')

    ax2.set_xlabel('Time', fontsize=18)
    ax2.set_ylabel('Rainfall ($mmhr^{-1}$)', fontsize=18)
    ax1.set_ylabel('outflow ($m^{3}s^{-1}$)', fontsize=18)

    ax2.legend().remove()

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    handles = handles1 + handles2
    labels = labels1 + labels2
    ax.legend(handles, labels, loc='center right', fontsize=15)

    # set x-axis label format to %d/%m %H
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %H:%M'))
    ax.tick_params(axis='x', labelsize=18, rotation=45)
    ax1.tick_params(axis='y', labelsize=18) ; ax2.tick_params(axis='y', labelsize=18)
    ax1.grid()


    return fig


In [6]:
def stat(storm_df):
    """
    Format statistical data from storm_df DataFrame.
    
    Calculates various statistical parameters based on the storm data in the storm_df DataFrame.
    
    Args:
        storm_df (pandas.DataFrame): DataFrame containing storm data from SWMM and OBS.
        
    Returns:
        Tuple containing the following:
        
        - swmm_total_outflow (float): Total outflow volume in cubic meters calculated from SWMM data.
        - obs_total_outflow (float): Total outflow volume in cubic meters calculated from OBS data.
        - swmm_max_outflow (float): outflow peak in CMS (Cubic Meters per Second) from SWMM data.
        - obs_max_outflow (float): outflow peak in CMS (Cubic Meters per Second) from OBS data.
        - swmm_max_outflow_time (datetime): Time of the outflow peak from SWMM data.
    """
    
    # Total outflow volume [cubic meter]
    swmm_total_outflow = sum(storm_df['SWMM outflow [CMS]'])
    obs_total_outflow = sum(storm_df['OBS outflow [CMS]'])

    # outflow peak [CMS] and time
    swmm_max_outflow = storm_df['SWMM outflow [CMS]'].max()
    swmm_max_outflow_time = storm_df[storm_df['SWMM outflow [CMS]'] == swmm_max_outflow].index[0]

    obs_max_outflow = storm_df['OBS outflow [CMS]'].max()
    obs_max_outflow_time = storm_df[storm_df['OBS outflow [CMS]'] == obs_max_outflow].index[0]

    return swmm_total_outflow*60*5, obs_total_outflow, swmm_max_outflow, obs_max_outflow


In [7]:
def update_imperviousness_with_factor(subcatchment_dict, factor):
    """
    Update the imperviousness values of SubCatchment objects in a dictionary by a factor.

    Args:
        subcatchment_dict (dict): A dictionary containing SubCatchment objects as values with subcatchment names as keys.
        factor (float): The factor by which imperviousness values need to be powered.

    Returns:
        dict: The updated dictionary with imperviousness values powered by the factor.
    """
#     impervious_initial_values = [21, 43, 47, 47, 51, 47, 51, 47, 43, 22, 37, 18, 24, 37, 47, 35, 45, 71, 19]
    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, subcatchment) in enumerate(subcatchment_dict.items()):
#         initial_imperviousness = impervious_initial_values[index]
#         subcatchment.imperviousness = initial_imperviousness
        powered_imperviousness = subcatchment.imperviousness * factor
        # Update the imperviousness value for the current subcatchment with the powered imperviousness value
        subcatchment.imperviousness = powered_imperviousness


    # Return the updated dictionary
    return subcatchment_dict


def update_storage_with_factor(subareas_dict, factor):
    """
    Update the storage values of SubArea objects in a dictionary by a factor.

    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        factor (float): The factor by which storage values need to be powered.

    Returns:
        dict: The updated dictionary with storage values powered by the factor.
    """
    # Loop through each subarea in the dictionary
    for subarea_name, subarea in subareas_dict.items():
        # Access the impervious and pervious storage values for the current subarea
        imp_storage = subarea.storage_imperv
        perv_storage = subarea.storage_perv
        # Multiply the impervious and pervious storage values by the factor
        powered_imp_storage = imp_storage * factor
        powered_perv_storage = perv_storage * factor
        # Update the storage values for the current subarea with the powered values
        subarea.storage_imperv = powered_imp_storage
        subarea.storage_perv = powered_perv_storage

    # Return the updated dictionary
    return subareas_dict

def update_n_with_factor(subareas_dict, factor):
    """
    Update the n_imperv and n_perv values of SubArea objects in a dictionary by a factor.

    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        factor (float): The factor by which n_imperv and n_perv values need to be multiplied.

    Returns:
        dict: The updated dictionary with n_imperv and n_perv values multiplied by the factor.
    """
    # Loop through each subarea in the dictionary
    for subarea_name, subarea in subareas_dict.items():
        # Access the impervious and pervious n values for the current subarea
        imp_n = subarea.n_imperv
        perv_n = subarea.n_perv
        # Multiply the impervious and pervious n values by the factor
        updated_imp_n = imp_n * factor
        updated_perv_n = perv_n * factor
        # Update the n values for the current subarea with the multiplied values
        subarea.n_imperv = updated_imp_n
        subarea.n_perv = updated_perv_n

    # Return the updated dictionary
    return subareas_dict


def update_width_with_factor(subcatchment_dict, factor, width_initial_values):
    """
    Update the width values of SubCatchment objects in a dictionary by a factor.

    Args:
        subcatchment_dict (dict): A dictionary containing SubCatchment objects as values with subcatchment names as keys.
        factor (float): The factor by which width values need to be powered.
        width_initial_values (numpy.ndarray): A NumPy array of initial width values corresponding to each subcatchment.

    Returns:
        dict: The updated dictionary with width values powered by the factor.
    """
    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, subcatchment) in enumerate(subcatchment_dict.items()):
        width = width_initial_values[index]  # Get the initial width value for the current subcatchment
        powered_width = width * factor
        subcatchment.width = powered_width

    # Return the updated dictionary
    return subcatchment_dict


def update_curve_num_with_factor(infiltration_dict, cn_factor):
    """
    Update the curve_no values of InfiltrationCurveNumber objects in a dictionary by a factor.

    Args:
        infiltration_dict (dict): A dictionary containing InfiltrationCurveNumber objects as values with subcatchment
                                  names as keys.
        cn_factor (float): The factor by which curve_no values need to be multiplied.

    Returns:
        dict: The updated dictionary with curve_no values multiplied by the factor.
    """
    # Loop through each subcatchment in the dictionary
    for subcatchment_name, infiltration_curve in infiltration_dict.items():
        # Get the initial curve_no value for the current subcatchment
        curve_no = int(infiltration_curve.curve_no)
        # Multiply the curve_no value by the factor
        powered_curve_no = curve_no * cn_factor
        # Update the curve_no value for the current subcatchment with the multiplied value
        infiltration_curve.curve_no = powered_curve_no

    # Return the updated dictionary
    return infiltration_dict

def update_pct_zero_with_factor(subareas_dict, pct_zero_factor):
    """
    Update the pct_zero values of SubArea objects in a dictionary by a factor.
    
    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        pct_zero_factor (float): The factor by which pct_zero values need to be multiplied.
        pct_zero_initial_values (numpy.ndarray): A NumPy array of initial pct_zero values corresponding to each subarea.
    
    Returns:
        dict: The updated dictionary with pct_zero values multiplied by the factor.
    """
    # Loop through each subarea in the dictionary
    for index, (subarea_name, subarea) in enumerate(subareas_dict.items()):
        # Get the initial pct_zero value for the current subarea
        pct_zero = subarea.pct_zero
        # Multiply the pct_zero value by the factor
        updated_pct_zero = pct_zero * pct_zero_factor
        # Update the pct_zero value for the current subarea with the multiplied value
        subarea.pct_zero = updated_pct_zero
    
    # Return the updated dictionary
    return subareas_dict

In [8]:
def Run_Model(date, imp_factor, storage_factor, n_factor, pct_zero_factor, cn_factor):

    # Load sim inpfile
    sim_dir = r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/"
    sim_dir = os.path.join(sim_dir, date + '/')
    INP_FILE = r"Final.inp"
    
    OBS_FILE = date 
    obs_outflow_df = data_5min_interpolate(load_outflow_obs_to_df(sim_dir + OBS_FILE))
    
    inp = read_inp_file(os.path.join(sim_dir, INP_FILE))

    # Select the relevant sections
    subcatchment_d = dict(inp[sections.SUBCATCHMENTS])
    infiltration_d = dict(inp[sections.INFILTRATION])
    subareas_d = dict(inp[sections.SUBAREAS])
    timeseries_d = dict(inp[sections.TIMESERIES])

    # Update the section value by factor
    update_imperviousness_with_factor(subcatchment_d, imp_factor)
    update_storage_with_factor(subareas_d, storage_factor)
#     update_width_with_factor(subcatchment_d, width_factor, width_initial_values)
    update_n_with_factor(subareas_d, n_factor)        
    update_pct_zero_with_factor(subareas_d, pct_zero_factor)
    update_curve_num_with_factor(infiltration_d, cn_factor)

    file_name = 'raanana_VB_norain'
    inp.write_file(os.path.join(sim_dir, file_name + '.inp'))
    swmm5_run(os.path.join(sim_dir, file_name + '.inp'), progress_size=1)
    OUT_FILE = file_name + '.out'

    sim_df = load_sim_to_df(os.path.join(sim_dir, OUT_FILE))
    
    storm_df = pd.concat([obs_outflow_df, sim_df], axis=1)
    storm_df[storm_df < 0] = 0
    storm_df = storm_df.fillna(0)
    swmm_total_outflow, obs_total_outflow, swmm_max_outflow, obs_max_outflow = stat(storm_df)
    
    return swmm_max_outflow


In [9]:
def generate_and_save_results(samples_num, imp_factor_range, storage_factor_range, n_factor_range, pct_zero_factor_range, cn_factor_range, sim_dir, date):
    factors_num = 5
    scenarios_num = 3
    val1 = np.random.rand(samples_num, factors_num)
    val2 = np.random.rand(samples_num, factors_num)
    results = []

    for factor_index in range(factors_num):
        result = np.zeros((samples_num, scenarios_num))
        for scenario_index in range(scenarios_num):
            if scenario_index == 0:
                val = val1.copy()
            elif scenario_index == 1:
                val = val2.copy()
                
                val[:, factor_index] = val1[:, factor_index]
            else:
                val = val1.copy()
                val[:, factor_index] = val2[:, factor_index]
            
#             print(scenario_index)
#             print(f'val: {val}')
#             print(f'val1: {val1}')
#             print(f'val2: {val2}')

            for sample_index in range(len(val)):
                imp_factor = imp_factor_range[0] + (imp_factor_range[1] - imp_factor_range[0]) * val[sample_index, 0]
                storage_factor = storage_factor_range[0] + (storage_factor_range[1] - storage_factor_range[0]) * val[sample_index, 1]
                n_factor = n_factor_range[0] + (n_factor_range[1] - n_factor_range[0]) * val[sample_index, 2]
                pct_zero_factor = pct_zero_factor_range[0] + (pct_zero_factor_range[1] - pct_zero_factor_range[0]) * val[sample_index, 3]
                cn_factor = cn_factor_range[0] + (cn_factor_range[1] - cn_factor_range[0]) * val[sample_index, 4]

                # Run the model with specific factors and save the result
                # Assuming Run_Model is defined elsewhere
                swmm_max_outflow = Run_Model(date, imp_factor, storage_factor, n_factor, pct_zero_factor, cn_factor)
                result[sample_index, scenario_index] = swmm_max_outflow
        results.append(result)
        
    return results



## Load Data

In [10]:
# Raanana sub-basin shapefile
raanana_basins_shapfile = r'D:\Development\RESEARCH\Raanana\gis\GIS\28_subcatchments\raanana_28_subcatchments.shp'
raanana_basins_gdf = gpd.read_file(raanana_basins_shapfile)  # sub-basins poly
raanana_basins_gdf.drop(columns=['Shape_Area', 'Area_km2', 'Area_ha','Area_m2'], inplace = True)

# Load the storm MAT file
radar_dir = r'D:\Development\RESEARCH\Raanana\data\rain_radar\Row_radar_correct_MAT_files/'
EVENTS_AND_BIAS_L = ['20120113', '20160108', '20180101', '20191213']


## SELECT the storm event:
event_idx = 0

## Load rain array
filename_date = EVENTS_AND_BIAS_L[event_idx]
radar_data = sio.loadmat(radar_dir + filename_date)
date = '_'.join([filename_date[:4], filename_date[4:6], filename_date[6:]])

## Run the simulations and save as pickles

In [11]:
sim_dir = r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/"
sim_dir = os.path.join(sim_dir, date, 'pickles' + '/')
num_runs = 50
samples_num = 5
tot_samples_num = num_runs * samples_num
print(f'Total number of samples: {tot_samples_num}')

for num_run in range(num_runs):
    # Define the ranges for each factor
    imp_factor_range = (0.1, 1.35)  # cant be bigger than 1.39
    storage_factor_range = (0.2, 10)
    n_factor_range = (0.5, 3)
    pct_zero_factor_range = (0.1, 1.4)  # cant be bigger than 1.44
    cn_factor_range = (0.5, 2.5)  # cant be bigger than 2.56

    # Check if the file already exists
    file_path = os.path.join(sim_dir, f'results_VB_Norain_{num_run}.pickle')
    if os.path.exists(file_path):
        print(f'File {file_path} already exists. Moving to the next num_run.')
        continue

    # Generate and save the results
    results = generate_and_save_results(
        samples_num, imp_factor_range, storage_factor_range,
        n_factor_range, pct_zero_factor_range, cn_factor_range,
        sim_dir, date
    )
#     print(results)
#     Save the results as a pickle file
    with open(file_path, 'wb') as f:
        pickle.dump(results, f)

    print(f'Saved results_VB_Norain{num_run}.pickle in {sim_dir}')


Total number of samples: 250
File D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13\pickles/results_VB_Norain_0.pickle already exists. Moving to the next num_run.


swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13/raanana_VB_norain.inp:  …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13/raanana_VB_norain.inp:  …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13/raanana_VB_norain.inp:  …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13/raanana_VB_norain.inp:  …

...

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13/raanana_VB_norain.inp:  …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13/raanana_VB_norain.inp:  …

Saved results_VB_Norain22.pickle in D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13\pickles/


swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2012_01_13/raanana_VB_norain.inp:  …

KeyboardInterrupt: 

In [ ]:
np.array(results).shape
results

In [ ]:
# %whos
